In [1]:
import pandas as pd
import numpy as np
import requests
import streamlit as st
from datetime import timedelta



In [2]:
# Configuration and Constants
OPENAQ_MEASUREMENTS = "https://api.openaq.org/v2/measurements"
OPENWEATHER_URL = "https://api.openweathermap.org/data/2.5/weather"
CITY = "Delhi"
PARAM = "pm25"

In [3]:
AQI_BREAKPOINTS = [
    (0.0, 12.0, "Good"),
    (12.1, 35.4, "Moderate"),
    (35.5, 55.4, "Unhealthy for Sensitive Groups"),
    (55.5, 150.4, "Unhealthy"),
    (150.5, 250.4, "Very Unhealthy"),
    (250.5, 500.4, "Hazardous"),
]

In [4]:
def pm25_to_aqi(pm):
    #converting pm2.5 to aqi category
    for low, high, category in AQI_BREAKPOINTS:
         if low <= pm <= high:
             return category
    return "hazardous"  # Default to highest category if out of range


In [8]:
@st.cache_data(ttl=600)
def fetch_openaq(city= CITY, parameter= PARAM, limit=10000):
      params = {
            "city": CITY,
            "parameter": parameter,
            "limit": limit,
            "sort": "desc",
            "order_by" : "datetime"
      }

      r = requests.get(OPENAQ_MEASUREMENTS, params=params, timeout=30)
      r.raise_for_status()
      weather_result = r.json().get("results", [])
      if len(weather_result)==0:
            st.warning("No data found from OpenAQ API")
            return pd.DataFrame()   
      df = pd.DataFrame(weather_result)
      df['datetime_utc'] = pd.to_datetime(df['date.utc'])
      return df


2025-10-23 18:45:24.439 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [ ]:
def preprocess(df_raw, openweather_api_key):
      if df_raw.empty:
            return df_raw
      df = df_raw.copy()
      df = df[['datetime_utc', 'value']]
      df = df.rename(columns={'value': 'pm25'})
      df['aqi_category'] = df['pm25'].apply(pm25_to_aqi)
      df['date'] = df['datetime_utc'].dt.date

      # Fetch weather data for each unique date
      unique_dates = df['date'].unique()
      weather_data = {}
      for date in unique_dates:
            weather_data[date] = fetch_weather_for_date(date, openweather_api_key)

      # Map weather data to the main dataframe
      df['temperature'] = df['date'].map(lambda d: weather_data[d]['temp'] if d in weather_data else np.nan)
      df['humidity'] = df['date'].map(lambda d: weather_data[d]['humidity'] if d in weather_data else np.nan)

      return df